# Tutorial 01 — Declare a property model *once*

`fairfluids.analysis.models` (alias **`fm`**) is the single source of truth for every property
model in the framework. A model is **one sympy expression** whose free symbols are classified into
three roles:

* **features** — data columns supplied per group (`T`, `p`, `x` …);
* **constants** — fixed before the fit (a literal, or resolved from each group's data);
* **parameters** — everything else: the quantities to fit / sample.

Both fitting backends (`fit.fit_least_squares` for SciPy and `fit.fit_mcmc` for NumPyro) derive
their kernels from this one object, so a model can **never drift** between the frequentist and
Bayesian pipelines.

This tutorial only needs sympy + numpy — no JAX/NumPyro.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import numpy as np
import pandas as pd
import sympy as sp

# Absolute data paths so the notebook runs from anywhere.
ROOT = "/home/sga/Code/FAIRFluids"
DENSITY = os.path.join(ROOT, "transition_water", "data", "Density")
VISCOSITY = os.path.join(ROOT, "transition_water", "data", "Viscosity")

from fairfluids.analysis import models as fm

print("built-in models:")
for name in fm.list_models():
    m = fm.get_model(name)
    print(f"  {name:38s}  property={m.property:9s}  params={m.param_names}")

built-in models:
  arrhenius                               property=viscosity  params=('Ea', 'logA')
  density_exp_poly                        property=density    params=('A1', 'A2', 'rho0')
  density_exp_poly_degC_anchored          property=density    params=('A1', 'A2')
  density_exp_poly_t0                     property=density    params=('A1', 'A2', 'rho0')
  density_exp_poly_t0_anchored            property=density    params=('A1', 'A2')
  density_exp_poly_t0_mean_centered       property=density    params=('A1', 'A2')
  extended_arrhenius                      property=viscosity  params=('Ea', 'lnB', 'n')
  litovitz                                property=viscosity  params=('a', 'b')
  litovitz_extended                       property=viscosity  params=('a', 'b', 'n')
  vft                                     property=viscosity  params=('B', 'T0', 'ln_eta0')


## 1. Inspect a built-in model

The framework ships a small library (viscosity: Arrhenius/VFT/Litovitz; density: exp-poly variants).
Every model exposes the same introspection surface.

In [2]:
vft = fm.get_model("vft")
print("name        :", vft.name)
print("expr        :", vft.expr)          # natural form: exp(A + B/(T - T0))
print("mean_expr   :", vft.mean_expr)     # observation scale (ln applied): A + B/(T - T0)
print("features    :", vft.features)
print("parameters  :", vft.param_names)
print("constants   :", vft.constant_names)
print("arg_order   :", vft.arg_order)     # canonical kernel arg order
print("param_units :", dict(vft.param_units))
print("derived     :", vft.derived_names)

name        : vft
expr        : exp(B/(T - T0) + ln_eta0)
mean_expr   : B/(T - T0) + ln_eta0
features    : ('T',)
parameters  : ('B', 'T0', 'ln_eta0')
constants   : ()
arg_order   : ('T', 'B', 'T0', 'ln_eta0')
param_units : {'ln_eta0': None, 'B': 'K', 'T0': 'K'}
derived     : ('eta0',)


## 2. Declare your own model

`fm.define_model` classifies the symbols for you. Anything that is neither a declared *feature* nor
a declared *constant* becomes a fitted *parameter*. Pass `overwrite=True` so re-running the cell is
idempotent.

In [3]:
T, Ea, logA = sp.symbols("T Ea logA")
R = 8.314

arr = fm.define_model(
    "tut_arrhenius", property="viscosity",
    expr=sp.exp(logA + Ea / (R * T)), features=["T"],
    p0={"logA": -10.0, "Ea": 15000.0},
    param_units={"logA": None, "Ea": "J/mol"},
    overwrite=True,
)
print("features   :", arr.features)
print("parameters :", arr.param_names)   # ('Ea', 'logA') — sorted, role-classified
print("mean_expr  :", arr.mean_expr)

features   : ('T',)
parameters : ('Ea', 'logA')
mean_expr  : 0.120279047389945*Ea/T + logA


## 3. Constants resolved from the data

A *constant* is a symbol fixed before fitting. Three resolver kinds ship out of the box:

| kind | meaning |
|---|---|
| `fixed` | a literal value (or just pass a number) |
| `mean` | the arithmetic mean of a feature column, e.g. `T0 = mean(T)` |
| `interp` | the observation linearly interpolated at an anchor, e.g. `rho0 = rho(mean T)` |

Declaring `T0` and `rho0` as constants leaves only the curvature coefficients to fit.

In [5]:
T, T0, rho0, A1, A2 = sp.symbols("T T0 rho0 A1 A2")
rho_expr = rho0 * sp.exp(-(A1 * (T - T0) + sp.Rational(1, 2) * A2 * (T**2 - T0**2)))

rho_model = fm.define_model(
    "tut_rho_centered", property="density", expr=rho_expr, features=["T"],
    constants={"T0":  {"kind": "mean",   "feature": "T"},
               "rho0": {"kind": "interp", "feature": "T"}},
    p0={"A1": 7e-4, "A2": 1e-6}, overwrite=True,
)
print("parameters :", rho_model.param_names)     # only A1, A2 fitted
print("constants  :", rho_model.constant_names)  # T0, rho0 resolved per group

# Resolve them by hand to see what a fit would compute internally:
from fairfluids.analysis.models.resolvers import resolve_constants
T_arr = np.linspace(280.0, 340.0, 7)
rho_arr = 1000.0 - 0.4 * (T_arr - 298.15)
consts = resolve_constants(rho_model.constants, {"T": T_arr}, rho_arr)
print("resolved   :", {k: round(v, 4) for k, v in consts.items()})

parameters : ('A1', 'A2')
constants  : ('T0', 'rho0')
resolved   : {'T0': 310.0, 'rho0': 995.26}


## 4. Derived quantities — reparametrisations in one declarative place

`derived=` lets you compute quantities *from* the fitted parameters/constants (e.g. activation
energy in kJ/mol, or a pre-exponential factor). The least-squares backend evaluates these **and
propagates their uncertainty** via the parameter covariance (delta method), so you never hand-derive
error formulas.

In [4]:
T, Ea, logA = sp.symbols("T Ea logA")
R = 8.314
arr2 = fm.define_model(
    "tut_arrhenius_derived", property="viscosity",
    expr=sp.exp(logA + Ea / (R * T)), features=["T"],
    p0={"logA": -10.0, "Ea": 15000.0},
    derived={"Ea_kJ_mol": "Ea / 1000", "A": "exp(logA)"},
    derived_units={"Ea_kJ_mol": "kJ/mol", "A": "Pa*s"},
    overwrite=True,
)
print("derived names :", arr2.derived_names)
print("derived exprs :", {k: str(v) for k, v in arr2.derived_exprs.items()})

derived names : ('A', 'Ea_kJ_mol')
derived exprs : {'Ea_kJ_mol': 'Ea/1000', 'A': 'exp(logA)'}


## 5. Compile the mean to numpy *and* JAX — same maths

`compile_numpy` / `compile_jax` lambdify `mean_expr` with the canonical `arg_order`
(features, then constants, then parameters). The two backends share this expression, so they agree
to floating-point.

In [6]:
np_kernel = fm.compile_numpy(arr)
jx_kernel = fm.compile_jax(arr)            # imports jax lazily

T_eval = np.array([280.0, 300.0, 320.0])
# arg order for arr: features (T), then params sorted (Ea, logA)
y_np = np_kernel(T_eval, 15000.0, -10.0)
y_jx = np.asarray(jx_kernel(T_eval, 15000.0, -10.0))
print("numpy mean_expr :", np.round(y_np, 6))
print("jax   mean_expr :", np.round(y_jx, 6))
print("max abs diff    :", float(np.max(np.abs(y_np - y_jx))))

numpy mean_expr : [-3.55648  -3.986048 -4.36192 ]
jax   mean_expr : [-3.55648  -3.986048 -4.36192 ]
max abs diff    : 0.0


## 6. JSON round-trip — author once, persist, reload

Models serialise to a `{"models": [...]}` file. The expression is stored as a **math string**
parsed back through a safe whitelist (no arbitrary code). This is the exact same shape as the
built-in `models/store/*.json` files.

In [7]:
import tempfile, json

path = os.path.join(tempfile.gettempdir(), "tut01_models.json")
fm.save_models([arr2, rho_model], path)

reloaded = {m.name: m for m in fm.load_models(path)}
rt = reloaded["tut_rho_centered"]
print("reloaded         :", list(reloaded))
print("expr preserved   :", sp.simplify(rt.expr - rho_model.expr) == 0)
print("roles preserved  :", rt.param_names, rt.constant_names, rt.features)

# Peek at the on-disk form of one model:
on_disk = json.loads(open(path).read())["models"][0]
print("stored expr      :", on_disk["expr"])
print("stored constants :", on_disk["constants"])

reloaded         : ['tut_arrhenius_derived', 'tut_rho_centered']
expr preserved   : True
roles preserved  : ('A1', 'A2') ('T0', 'rho0') ('T',)
stored expr      : exp(0.120279047389945*Ea/T + logA)
stored constants : {}


## Recap

* `fm.define_model` turns one sympy expression into a `SymbolicModel`, auto-classifying symbols
  into **features / constants / parameters**.
* **Constants** can be literals or resolved per-group (`mean`, `interp`) — this is how
  data-anchored density models stay one-liners.
* **Derived quantities** centralise reparametrisations; their uncertainty is propagated for you.
* `compile_numpy` / `compile_jax` guarantee the two backends share identical maths.
* `fm.save_models` / `fm.load_models` make declarations portable.

**Next:** `02_frequentist_fitting.ipynb` — fit these models with SciPy.